In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data source: https://www.kaggle.com/datasets/anandshaw2001/customer-churn-dataset

# Load the dataset
df = pd.read_csv("Churn_Modelling.csv")

# Rename target columns
df = df.rename(columns={
    'Exited': 'churn'
})

# Features and target
X = df[
    [
        'CreditScore',
        'Geography',
        'Gender',
        'Age',
        'Tenure',
        'Balance',
        'NumOfProducts',
        'HasCrCard',
        'IsActiveMember',
        'EstimatedSalary'
    ]
]

y = df['churn']

# Numerical and categorical features
numerical_features = [
    'CreditScore',
    'Age',
    'Tenure',
    'Balance',
    'NumOfProducts',
    'HasCrCard',
    'IsActiveMember',
    'EstimatedSalary'
]

categorical_features = [
    'Geography',
    'Gender'
]

# Preprocessing: Scale numerical features and encode categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_features
        ),
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    (
        'classifier',
        LogisticRegression(
            random_state=42,
            max_iter=1000
        )
    )
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train model
model.fit(X_train, y_train)

# Predict churn probability for a new customer
new_customer = pd.DataFrame({
    'CreditScore': [650],
    'Geography': ['France'],
    'Gender': ['Female'],
    'Age': [40],
    'Tenure': [5],
    'Balance': [80000],
    'NumOfProducts': [1],
    'HasCrCard': [1],
    'IsActiveMember': [0],
    'EstimatedSalary': [75000]
})

churn_probability = model.predict_proba(new_customer)[0][1]

# Classify based on threshold
threshold = 0.5

churn_prediction = (
    1 if churn_probability >= threshold else 0
)

print(f"Churn Probability for new customer: {churn_probability:.2%}")
print(
    f"Churn Prediction "
    f"(1 = churn, 0 = no churn): {churn_prediction}"
)

# Display model coefficients
categorical_feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_features)
    .tolist()
)

feature_names = numerical_features + categorical_feature_names

coefficients = model.named_steps['classifier'].coef_[0]

print("\nModel Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.4f}")

Churn Probability for new customer: 28.12%
Churn Prediction (1 = churn, 0 = no churn): 0

Model Coefficients:
CreditScore: -0.0860
Age: 0.7392
Tenure: -0.0202
Balance: 0.1609
NumOfProducts: -0.0702
HasCrCard: -0.0321
IsActiveMember: -0.5158
EstimatedSalary: 0.0478
Geography_France: -0.5699
Geography_Germany: 0.2535
Geography_Spain: -0.5236
Gender_Female: -0.1579
Gender_Male: -0.6821
